# **AI ASSIGNMENT # 02**
## **I242533**
## **Dawar Ahmed** 
## **DS - *B***

## GitHub Repository
[View the complete project on GitHub](https://github.com/DawarAhmed27/i242533_DawarAhmed_A2/blob/main/i242533_DawarAhmed_A2.ipynb)

*Note: Update the repository link with your actual GitHub username and repository URL*

In [77]:
import random

#Card Class
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = str(value)

    def __repr__(self):
        return f"{self.color} {self.value}"
    
    def matches(self, other_card):
        return self.color == other_card.color or self.value == other_card.value
#DeckGenerator
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    numbers = list(range(10))
    
    deck = []
    for color in colors:
        for num in numbers:
            deck.append(Card(color, num))
        #Adding Skip cards
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))
        
    random.shuffle(deck)
    return deck

#LegalMoveGenerator
def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        if card.matches(top_card):
            valid_moves.append(card)
    return valid_moves

#TestingStep1
deck = generate_deck()
player1_hand = [deck.pop() for _ in range(5)]
top_card = Card('Red', '5')

print("Top Card on Table:", top_card)
print("Player 1 Hand:", player1_hand)

Top Card on Table: Red 5
Player 1 Hand: [Blue 4, Green 3, Green 0, Blue 8, Green 5]


In [78]:
#GameStateClass
class GameState:
    def __init__(self, ai_cards, opponent1_card_count, opponent2_card_count, top_card, deck):
        self.ai_cards = ai_cards
        self.opponent1_cards = opponent1_card_count
        self.opponent2_cards = opponent2_card_count
        self.top_card = top_card
        self.deck = deck if isinstance(deck, list) else []
        
    def __repr__(self):
        return f"Top: {self.top_card} | AI Cards: {len(self.ai_cards)} | Opp1: {self.opponent1_cards} | Opp2: {self.opponent2_cards} | Deck: {len(self.deck)}"
#EvaluationFunction
def evaluate_state(state, strategy="Defensive"):
    c_ai = len(state.ai_cards)
    c_opp = (state.opponent1_cards + state.opponent2_cards) / 2.0
    s_count = sum(1 for card in state.ai_cards if card.value == 'Skip')
    score = 50 - (5 * c_ai) + (2 * c_opp) + (3 * s_count)
    if strategy == "Defensive":
        if state.opponent1_cards <= 1 or state.opponent2_cards <= 1:
            score -= 20
        score += (2 * s_count)
    elif strategy == "Offensive":
        if c_ai <= 1:
            score += 20
        score -= (2 * c_ai)
    return score
#TestingStep2
fake_ai_hand = [Card('Red', '5'), Card('Blue', 'Skip'), Card('Green', '9')]
fake_state = GameState(fake_ai_hand, 2, 4, Card('Red', '3'), 30)

 ## Evaluation Function Explanation

Formula: Score = 50 - 5*C_AI + 2*C_Opp + 3*S

Variables:
- C_AI = cards in our hand (want this LOW)
- C_Opp = average opponent hand size (want this HIGH)
- S = skip cards we have

How it works:
- Start with 50 points
- Lose 5 points for each card in our hand (goal is 0 cards to win)
- Gain 2 points for each card opponents have
- Gain 3 points for each Skip card

Defensive Strategy (Minimax/P1):
- Extra -20 penalty if opponent is about to win
- Extra +2 bonus for Skip cards
- Goal: Don't lose

Offensive Strategy (Expectimax/P2):
- Extra +20 bonus if we're close to winning
- Extra -2 penalty if we hold many cards
- Goal: Win fast

In [79]:
#SimulateMoveFunction
def simulate_move(state, move, player_turn):
    new_ai_cards = state.ai_cards.copy()
    new_opponent1_cards = state.opponent1_cards
    new_opponent2_cards = state.opponent2_cards
    new_top_card = state.top_card
    new_deck = state.deck.copy() if isinstance(state.deck, list) else []
    if player_turn == 0:
        if move == "Draw":
            if len(new_deck) > 0:
                drawn_card = new_deck.pop(0)
                new_ai_cards.append(drawn_card)
        else:
            for c in new_ai_cards:
                if c.color == move.color and c.value == move.value:
                    new_ai_cards.remove(c)
                    new_top_card = move
                    break
    elif player_turn == 1:
        if move == "Draw":
            if len(new_deck) > 0:
                new_deck.pop(0)
                new_opponent1_cards += 1
        else:
            new_opponent1_cards -= 1
            new_top_card = move
    elif player_turn == 2:
        if move == "Draw":
            if len(new_deck) > 0:
                new_deck.pop(0)
                new_opponent2_cards += 1
        else:
            new_opponent2_cards -= 1
            new_top_card = move
    return GameState(new_ai_cards, new_opponent1_cards, new_opponent2_cards, new_top_card, new_deck)

#TestingStep3
#TestingStep3
test_deck=[Card('Red', '1'), Card('Blue', '2'), Card('Green', '3'), Card('Yellow', '5')]
test_state = GameState([Card('Red', '5'), Card('Blue', 'Skip')], 3, 2, Card('Red', '3'), test_deck)
print("Test 1 AI plays card:", simulate_move(test_state, Card('Red', '5'), 0))
print("Test 2 AI draws card:", simulate_move(test_state, "Draw", 0))


Test 1 AI plays card: Top: Red 5 | AI Cards: 1 | Opp1: 3 | Opp2: 2 | Deck: 4
Test 2 AI draws card: Top: Red 3 | AI Cards: 3 | Opp1: 3 | Opp2: 2 | Deck: 3


In [80]:
#MinimaxAlgorithm(DefensiveAI)
def minimax(state, depth, alpha, beta, player_turn):
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Defensive"), None
        
    if player_turn == 0:
        max_eval = float('-inf')
        best_move = None
        
        valid_moves = get_valid_moves(state.ai_cards, state.top_card)
        if not valid_moves:
            valid_moves = ["Draw"]
            
        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, 1)
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
                
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break
        return max_eval, best_move
    else:
        min_eval = float('inf')
        possible_moves = [state.top_card, "Draw"]
        for move in possible_moves:
            new_state = simulate_move(state, move, player_turn)
            next_turn = 2 if player_turn == 1 else 0
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, next_turn)
            if eval_score < min_eval:
                min_eval = eval_score
            beta = min(beta, eval_score)
            if beta <= alpha:
                break
        return min_eval, None
#TestingStep4
best_score, best_move = minimax(test_state, 3, float('-inf'), float('inf'), 0)
print("Minimax Best Move:", best_move)

Minimax Best Move: Red 5


In [81]:
#ExpectimaxAlgorithm(OffensiveAI)
def expectimax(state, depth, player_turn):
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Offensive"), None
        
    if player_turn == 0:
        max_eval = float('-inf')
        best_move = None
        
        valid_moves = get_valid_moves(state.ai_cards, state.top_card)
        if not valid_moves:
            valid_moves = ["Draw"]
            
        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = expectimax(new_state, depth - 1, 1)
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
        return max_eval, best_move
    else:
        expected_eval = 0
        possible_moves = [state.top_card, "Draw"]
        for move in possible_moves:
            if move == "Draw" and isinstance(state.deck, list) and len(state.deck) > 0:
                deck_len = len(state.deck)
                draw_expected = 0
                for drawn_card in state.deck:
                    probability = 1.0 / deck_len
                    if drawn_card.matches(state.top_card):
                        temp_deck = [c for c in state.deck if c != drawn_card]
                        if player_turn == 1:
                            temp_state = GameState(state.ai_cards, state.opponent1_cards + 1, state.opponent2_cards, drawn_card, temp_deck)
                        else:
                            temp_state = GameState(state.ai_cards, state.opponent1_cards, state.opponent2_cards + 1, drawn_card, temp_deck)
                    else:
                        temp_deck = [c for c in state.deck if c != drawn_card]
                        if player_turn == 1:
                            temp_state = GameState(state.ai_cards, state.opponent1_cards + 1, state.opponent2_cards, state.top_card, temp_deck)
                        else:
                            temp_state = GameState(state.ai_cards, state.opponent1_cards, state.opponent2_cards + 1, state.top_card, temp_deck)
                    next_turn = 2 if player_turn == 1 else 0
                    eval_score, _ = expectimax(temp_state, depth - 1, next_turn)
                    draw_expected += probability * eval_score
                expected_eval += draw_expected
            else:
                new_state = simulate_move(state, move, player_turn)
                next_turn = 2 if player_turn == 1 else 0
                eval_score, _ = expectimax(new_state, depth - 1, next_turn)
                expected_eval += eval_score
        expected_eval = expected_eval / len(possible_moves)
        return expected_eval, None
#TestingStep5
best_exp_score, best_exp_move = expectimax(test_state, 3, 0)
print("Expectimax Best Move:", best_exp_move)

Expectimax Best Move: Red 5


In [83]:
#PlayerInputHandler
def get_manual_move(hand, top_card):
    """Get a valid move from player 3 (manual mode)"""
    while True:
        print(f"\nTop Card: {top_card}")
        print("Your Hand:")
        for i, card in enumerate(hand, 1):
            print(f"  {i}. {card}")
        print("  0. Draw")
        choice=input("Enter your choice (0 to draw, or card number): ").strip()
        if choice=="0":
            return "Draw"
        try:
            card_idx=int(choice)-1
            if card_idx<0 or card_idx>=len(hand):
                print("Invalid selection. Please choose a valid card number.")
                continue
            chosen_card=hand[card_idx]
            if chosen_card.matches(top_card):
                return chosen_card
            else:
                print("Invalid move!", chosen_card, "does not match", top_card)
        except ValueError:
            print("Invalid input. Please enter a number.")

#FinalGameLoopSimulation
def play_uno(p3_mode="simulation"):
    print("Starting UNO AI Game")
    deck = generate_deck()
    p1_hand = [deck.pop() for _ in range(5)]
    p2_hand = [deck.pop() for _ in range(5)]
    p3_hand = [deck.pop() for _ in range(5)]
    
    top_card = deck.pop()
    current_player = 0
    while True:
        if len(p1_hand) == 0:
            print("Player 1 Wins")
            break
        elif len(p2_hand) == 0:
            print("Player 2 Wins")
            break
        elif len(p3_hand) == 0:
            print("Player 3 Wins")
            break
        if len(deck) == 0:
            print("Draw Deck Empty")
            break
        print("Top card:", top_card)
        if current_player == 0:
            print("Player 1 Defensive AI Turn")
            print("P1 hand:", p1_hand)
            state = GameState(p1_hand, len(p2_hand), len(p3_hand), top_card, deck)
            score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
            print("P1 decisions considered at depth 1:")
            valid_moves = get_valid_moves(p1_hand, top_card)
            if not valid_moves:
                valid_moves = ["Draw"]
            for v_move in valid_moves:
                test_state = simulate_move(state, v_move, 0)
                eval_score, _ = minimax(test_state, 2, float('-inf'), float('inf'), 1)
                print("Move:", v_move, "Score:", eval_score)
            if move == "Draw" or move is None:
                print("Action: P1 Draws a card")
                p1_hand.append(deck.pop())
                current_player = 1
            else:
                print("Action: P1 Plays", move)
                for c in p1_hand:
                    if c.color == move.color and c.value == move.value:
                        p1_hand.remove(c)
                        break
                top_card = move
                if top_card.value == 'Skip':
                    print("Player 2 Skipped")
                    current_player = 2
                else:
                    current_player = 1
        elif current_player == 1:
            print("Player 2 Offensive AI Turn")
            print("P2 hand:", p2_hand)
            state = GameState(p2_hand, len(p3_hand), len(p1_hand), top_card, deck)
            score, move = expectimax(state, 3, 0)
            print("P2 decisions considered at depth 1:")
            valid_moves = get_valid_moves(p2_hand, top_card)
            if not valid_moves:
                valid_moves = ["Draw"]
            for v_move in valid_moves:
                test_state = simulate_move(state, v_move, 0)
                eval_score, _ = expectimax(test_state, 2, 1)
                print("Move:", v_move, "Score:", eval_score)
            if move == "Draw" or move is None:
                print("Action: P2 Draws a card")
                p2_hand.append(deck.pop())
                current_player = 2
            else:
                print("Action: P2 Plays", move)
                for c in p2_hand:
                    if c.color == move.color and c.value == move.value:
                        p2_hand.remove(c)
                        break
                top_card = move
                if top_card.value == 'Skip':
                    print("Player 3 Skipped")
                    current_player = 0
                else:
                    current_player = 2
        elif current_player == 2:
            if p3_mode == "manual":
                print("\n" + "="*50)
                print("PLAYER 3 - MANUAL MODE (Your Turn)")
                print("="*50)
                move = get_manual_move(p3_hand, top_card)
                if move == "Draw":
                    print("You drew a card")
                    if len(deck) > 0:
                        p3_hand.append(deck.pop())
                    current_player = 0
                else:
                    print(f"You played: {move}")
                    for c in p3_hand:
                        if c.color == move.color and c.value == move.value:
                            p3_hand.remove(c)
                            break
                    top_card = move
                    if top_card.value == 'Skip':
                        print("Player 1 Skipped")
                        current_player = 1
                    else:
                        current_player = 0
            else:
                print("Player 3 Simulation AI Turn")
                print("P3 hand:", p3_hand)
                state = GameState(p3_hand, len(p1_hand), len(p2_hand), top_card, deck)
                score, move = minimax(state, 3, float('-inf'), float('inf'), 0)
                if move == "Draw" or move is None:
                    print("Action: P3 Draws a card")
                    if len(deck) > 0:
                        p3_hand.append(deck.pop())
                    current_player = 0
                else:
                    print("Action: P3 Plays", move)
                    for c in p3_hand:
                        if c.color == move.color and c.value == move.value:
                            p3_hand.remove(c)
                            break
                    top_card = move
                    if top_card.value == 'Skip':
                        print("Player 1 Skipped")
                        current_player = 1
                    else:
                        current_player = 0

#GameStartWrapper
def start_game():
    print("\n" + "="*50)
    print("UNO GAME - PLAYER 3 MODE SELECTION")
    print("="*50)
    while True:
        print("\nChoose Player 3 mode:")
        print("  (1) Manual (Play as Player 3)")
        print("  (2) Simulation (AI)")
        choice=input("Enter your choice (1 or 2): ").strip()
        if choice=="1":
            play_uno(p3_mode="manual")
            break
        elif choice=="2":
            play_uno(p3_mode="simulation")
            break
        else:
            print("Invalid choice. Please enter 1 or 2.")

#Run the Game

### Conclusion: Minimax vs. Expectimax

Based on the UNO game simulation, **Expectimax** generally performs better and acts as a more realistic AI compared to Minimax.

**Why?**
1. **Nature of UNO:** UNO is a stochastic game (game of chance) with imperfect information. We do not know the hidden cards of our opponents, nor do we know which card will be drawn from the deck next.
2. **Minimax Limitation:** Minimax assumes the opponent is playing optimally to minimize our score. However, in UNO, opponents cannot play "perfectly" against us because they don't know our cards. Minimax prepares for the absolute worst-case scenario, which makes it overly defensive and often leads to sub-optimal, slow gameplay.
3. **Expectimax Advantage:** Expectimax uses chance nodes to model the opponents' unknown moves as probabilities. It calculates the *average* expected outcome rather than assuming the worst-case. This aligns perfectly with UNO, where players make moves based on their own hands rather than purely trying to block one specific person. Expectimax takes calculated risks and sheds cards much more efficiently.

In [84]:
#TreePrinter-VisualFormat
def print_game_tree_visual(state, depth=2, player="AI"):
    """Print game tree in visual format matching PDF"""
    moves=get_valid_moves(state.ai_cards, state.top_card)
    if not moves:
        moves=["Draw"]
    next_players={"AI":"P2","P1":"P2","P2":"P3","P3":"P1"}
    next_player=next_players.get(player,"P2")
    move_strs=[str(m)if m!="Draw"else"Draw"for m in moves]
    
    if len(moves)==1:
        print(f"          {player}\n            |\n            |\n        {move_strs[0]}\n            |\n          {next_player}")
        return
    
    col_w=18
    total_w=len(moves)*col_w
    print(" "*((total_w-len(player))//2)+player)
    
    branch=""
    for i in range(len(moves)):
        if i==0:
            branch+=" "*(col_w//2-1)+"/"
        elif i==len(moves)-1:
            branch+=" "*(col_w//2-1)+"\\"
        else:
            branch+=" "*(col_w//2-2)+"|"+" "
    print(branch)
    
    move_line=""
    for move in move_strs:
        pad=(col_w-len(move))//2
        move_line+=" "*pad+move+" "*(col_w-len(move)-pad)
    print(move_line)
    
    if depth>1:
        vert_line=""
        for i in range(len(moves)):
            vert_line+=" "*(col_w//2)+"|"+" "*(col_w//2)
        print(vert_line)
        
        player_line=""
        for i in range(len(moves)):
            pad=(col_w-len(next_player))//2
            player_line+=" "*pad+next_player+" "*(col_w-len(next_player)-pad)
        print(player_line)

#LinearTreeVersion - Enhanced with Depth Labels
def print_game_tree(state, depth=3, indent="", player_turn=0):
    # Terminal node: show evaluation score
    if depth == 0:
        score = evaluate_state(state, "Defensive")
        print(indent + "Eval:", score)
        return
    
    # Get valid moves
    moves = get_valid_moves(state.ai_cards, state.top_card)
    if not moves:
        moves = ["Draw"]
    
    # Determine node label by depth
    if depth == 3:
        label = "P1 (Minimax - MAX)"
    elif depth == 2:
        label = "P2 (Opponent)"
    elif depth == 1:
        label = "P3 (MIN)"
    else:
        label = "Terminal"
    
    # Print node label at root
    if indent == "":
        print(label)
    
    # Print all moves
    for i, move in enumerate(moves):
        is_last = (i == len(moves) - 1)
        symbol = "└" if is_last else "├"
        move_str = move if move == "Draw" else str(move)
        print(indent + symbol + "-- " + move_str)
        
        # Simulate and go deeper
        new_state = simulate_move(state, move, 0)
        next_indent = indent + ("  " if is_last else "| ")
        next_depth = depth - 1
        next_player = (player_turn + 1) % 3
        
        # Label next level
        if next_depth > 0:
            if next_depth == 2:
                next_label = "CHANCE" if move == "Draw" else "P2"
            elif next_depth == 1:
                next_label = "CHANCE" if move == "Draw" else "P3"
            else:
                next_label = ""
            
            if next_label:
                print(next_indent + "- " + next_label)
                deeper_indent = next_indent + "  "
            else:
                deeper_indent = next_indent
        else:
            deeper_indent = next_indent
        
        # Recurse
        print_game_tree(new_state, next_depth, deeper_indent, next_player) 

In [85]:
# Demonstrate Labeled Game Tree
print("GAME TREE - Minimax Algorithm (P1)")
print("=" * 40)

# Create test state
test_hand = [Card('Red', '5'), Card('Red', '7'), Card('Red', 'Skip')]
test_state = GameState(test_hand, 3, 2, Card('Red', '3'), [Card('Blue', '1'), Card('Green', '2'), Card('Yellow', '3')])

print("Top Card:", test_state.top_card)
print("P1 Hand:", test_hand)
print("P2 Cards: 3")
print("P3 Cards: 2")
print()

print("Game Tree (Depth 3):")
print_game_tree(test_state, depth=3, indent="", player_turn=0)

GAME TREE - Minimax Algorithm (P1)
Top Card: Red 3
P1 Hand: [Red 5, Red 7, Red Skip]
P2 Cards: 3
P3 Cards: 2

Game Tree (Depth 3):
P1 (Minimax - MAX)
├-- Red 5
| - P2
|   ├-- Red 7
|   | - P3
|   |   └-- Red Skip
|   |     Eval: 55.0
|   └-- Red Skip
|     - P3
|       └-- Red 7
|         Eval: 55.0
├-- Red 7
| - P2
|   ├-- Red 5
|   | - P3
|   |   └-- Red Skip
|   |     Eval: 55.0
|   └-- Red Skip
|     - P3
|       └-- Red 5
|         Eval: 55.0
└-- Red Skip
  - P2
    ├-- Red 5
    | - P3
    |   └-- Red 7
    |     Eval: 55.0
    └-- Red 7
      - P3
        └-- Red 5
          Eval: 55.0


## Minimax vs Expectimax for UNO

### How Minimax Works (P1 - Defensive)

Minimax thinks: "The opponent will play their best move to beat me"
- Plans for worst case scenario
- Assumes opponent plays perfectly
- Very defensive

Problem: Opponents don't know what cards we have, so they can't play perfectly. This makes Minimax too safe and too slow.

### How Expectimax Works (P2 - Offensive)

Expectimax thinks: "The opponent will draw a random card from the deck"
- Calculates probability for each possible card
- More aggressive playstyle
- Takes calculated risks

Better for UNO because: Opponents actually don't know our cards, they draw randomly, and they don't play perfectly.

### Why Expectimax Wins More

1. Hidden information - we don't see opponent cards
2. Random draws determine a lot of the game
3. Real players don't play perfectly (Expectimax doesn't assume they do)
4. Expectimax calculates probability for each drawn card

### Problems We Notice

Minimax:
- Too slow to play (always defensive)
- Doesn't take chances even when it's safe to do so
- Assumes perfect information we don't actually have

Expectimax:
- Costs more to compute (checks each card in deck)
- Works best with full deck (less accurate with small deck)
- Assumes cards are randomly distributed (reasonable for UNO)

### Bottom Line

Expectimax is better for UNO because the game has randomness and hidden information. Minimax works better for games like chess where all information is visible and there's no luck.